# Evaluación 2 — Bases aprendidas vs. Fourier
**Programación Científica 2026-1 · Universidad Nacional de Colombia**
ID personalizado: **1591**

Pregunta guía: ¿ayuda empezar con la base "correcta" (Fourier) en lugar de una base
aleatoria, al entrenar una red que reconstruye una señal?

## 0. Preparación

In [2]:
# --- Librerías ---
import numpy as np                       # utilidades numéricas fuera de JAX (indexado, argmax, etc.)
import jax                                # motor de diferenciación automática y compilación (jit)
import jax.numpy as jnp                   # numpy "diferenciable" que corre dentro de JAX
from jax import random, grad, jit, vmap   # random: manejo de semillas; grad: gradiente automático;
                                           # jit: compilación; vmap: vectorización sobre batches
import matplotlib.pyplot as plt           # graficas

# --- Semilla global ---
# Usamos el ID de la evaluación como semilla para que todo el notebook
# sea reproducible: cualquiera que lo corra de nuevo obtiene los mismos
# pesos iniciales y, por lo tanto, las mismas curvas de pérdida.
SEED = 1591
key = random.PRNGKey(SEED)

## Parte I — El fenómeno físico

En un medio **sin dispersión** todas las frecuencias viajan igual y una
onda mantiene su forma. En un medio **dispersivo**, distintas frecuencias
viajan a distinta velocidad. Los datos de esta evaluación vienen de un
medio dispersivo.

Cada modo tiene un número de onda entero k, y la regla del medio le
asigna una frecuencia:

$$\omega(k) = \sqrt{c^2 k^2 + \mu^2}$$

Si µ=0 las frecuencias son bajas; si µ>0, el término µ² empuja las
frecuencias hacia arriba. El parámetro µ define los tres regímenes
(`y` en los datos): 0 = sin dispersión, 1 = intermedia, 2 = fuerte.

Un sensor fijo mide la suma de los modos en el tiempo:

$$f(t) = \sum_j a_j \cos(\omega(k_j)\, t + \varphi_j)$$

Es decir: **cada señal es una suma de cosenos a frecuencias bien
definidas**, justo lo que una base de senos y cosenos representa con
pocos coeficientes. Esta idea es la que verificamos más adelante con
la capa de Fourier.